# 1. Set up

## 1.1 Kernel

- For training users, please select the kernel "training"
- After choosing your kernel, wait for a few minutes until the kernel initializes properly

## 1.2 Spark Session
A spark session named ```spark``` is already built for you based on the configuration of your chosen template

The AppName is the same name in the Ocean Spark Dashboard. It is good practice to monitor your notebook in the dashboard. The notebook status should be "Running" (blue circle)

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

# 2. Import packages

In [2]:
import os
import pandas as pd
from datetime import datetime

In [3]:
from ais import functions as af

In [4]:
import pyspark.sql.functions as F

In [5]:
import h3
import folium
from shapely.geometry import Polygon

# 3. Read AIS Data

AIS data and ship register data are stored in Amazon S3 buckets.
We prepared a sample dataset for training users, which is a subset of the AIS data.
The training dataset contains AIS records of January 2024.

There are two ways to read the AIS data.
- Set a basepath, and read the AIS data using spark.read()
- Use the ais package to read the AIS data

## 3.1 Use spark.read() 

import os
basepath = os.environ.get(
    "AIS_BASEPATH",
    "s3a://ais-data-142496269814/exact-earth-data/transformed/prod/"
)
basepath- Get the base path

In [6]:
import os
basepath = os.environ.get(
    "AIS_BASEPATH",
    "s3a://ais-data-142496269814/exact-earth-data/transformed/prod/"
)
basepath

's3a://ais-data-142496269814/exact-earth-data/transformed/prod/'

- Read ALL AIS for one date 

Read from path ```basepath + "year=<YYYY>/month=<mm>/day=<dd>"```

In [7]:
df1 = spark.read.option("basepath",basepath).parquet(basepath+"year=2024/month=01/day=01")

In [8]:
df1.limit(1).show(vertical=True, truncate=True)

-RECORD 0---------------------------------
 mmsi              | 259511000            
 message_type      | 1                    
 dt_insert_utc     | 2024-01-01 15:52:08  
 longitude         | 32.74504667          
 latitude          | 76.37381667          
 imo               | 8131453              
 vessel_name       | VIMA                 
 callsign          | LJBD                 
 vessel_type       | Fishing              
 vessel_type_code  | 30                   
 vessel_type_cargo | NULL                 
 vessel_class      | A                    
 length            | 78.0                 
 width             | 12.0                 
 flag_country      | Norway               
 flag_code         | 259                  
 destination       | TROMSO               
 eta               | 3211234              
 draught           | 0.0                  
 sog               | 6.7                  
 cog               | 181.9                
 rot               | 0.0                  
 heading   

- Read AIS data of a list of dates

In [9]:
#Read for one-week
#puth all date paths in a list
df_week = spark.read.parquet(*[basepath + "year=2024/month=01/day=01",
                          basepath + "year=2024/month=01/day=02",
                          basepath + "year=2024/month=01/day=03",
                          basepath + "year=2024/month=01/day=04",
                          basepath + "year=2024/month=01/day=05",
                          basepath + "year=2024/month=01/day=06",
                          basepath + "year=2024/month=01/day=07"]
                       )

In [10]:
#Create a date column and then count rows per date
df_week = df_week.withColumn('date', F.to_date("dt_insert_utc"))
df_week.groupBy('date').count().show()

+----------+--------+
|      date|   count|
+----------+--------+
|2024-01-03|25334731|
|2024-01-04|25605384|
|2024-01-05|25947938|
|2024-01-06|25750192|
|2024-01-07|26058066|
|2024-01-01|25196836|
|2024-01-02|24665640|
+----------+--------+



## 3.2 Use ais package

- Read AIS data of a certain day

In [11]:
start_date = datetime.fromisoformat("2024-01-01")
df2 = af.get_ais(spark,start_date)

In [12]:
df2.limit(1).show(vertical=True, truncate=True)

-RECORD 0---------------------------------
 message_type      | 1                    
 mmsi              | 259511000            
 dt_insert_utc     | 2024-01-01 15:52:08  
 longitude         | 32.74504667          
 latitude          | 76.37381667          
 imo               | 8131453              
 vessel_name       | VIMA                 
 callsign          | LJBD                 
 vessel_type       | Fishing              
 vessel_type_code  | 30                   
 vessel_type_cargo | NULL                 
 vessel_class      | A                    
 length            | 78.0                 
 width             | 12.0                 
 flag_country      | Norway               
 flag_code         | 259                  
 destination       | TROMSO               
 eta               | 3211234              
 draught           | 0.0                  
 sog               | 6.7                  
 cog               | 181.9                
 rot               | 0.0                  
 heading   

- read AIS data of a date range

NOTE: For complex calculations, it is recommended to read only maximum of 1 month of data.

In [13]:
start_date = datetime.fromisoformat("2024-01-01")
end_date = datetime.fromisoformat("2024-01-05")
df3 = af.get_ais(spark,start_date, end_date = end_date)

In [14]:
df3.withColumn("date",F.col("dt_insert_utc").cast("date")) \
        .groupby('date')  \
        .agg(F.count("eeid").alias("count")) \
        .orderBy("date") \
.show()

+----------+--------+
|      date|   count|
+----------+--------+
|2024-01-01|25196836|
|2024-01-02|24665640|
|2024-01-03|25334731|
|2024-01-04|25605384|
|2024-01-05|25947938|
+----------+--------+



- read AIS data of a list of dates

In [15]:
date_list = [datetime.fromisoformat("2024-01-01"),
             datetime.fromisoformat("2024-01-08"),
             datetime.fromisoformat("2024-01-15")]
date_list

[datetime.datetime(2024, 1, 1, 0, 0),
 datetime.datetime(2024, 1, 8, 0, 0),
 datetime.datetime(2024, 1, 15, 0, 0)]

In [16]:
df4 = af.get_ais(spark,date_list=date_list)

In [17]:
df4.withColumn("date",F.col("dt_insert_utc").cast("date")) \
        .groupby('date')  \
        .agg(F.count("eeid").alias("count")) \
        .orderBy("date") \
.show()

+----------+--------+
|      date|   count|
+----------+--------+
|2024-01-01|25196836|
|2024-01-08|25726842|
|2024-01-15|26216856|
+----------+--------+



# 4. How to get AIS data of a certain place

## 4.1 Get H3 Index

We can get the H3 index of any pair of latitude and longitude using geo_to_h3 function from h3 package.

In [18]:
#Eg:point in Suez Canal
latitude = 30.56529681740963
longitude = 32.33832930701776

In [19]:
#Resolution of H3 Index used in partitioning
h3index_0 = h3.geo_to_h3(latitude, longitude, 0) #resolution 0
print(h3index_0)
h3index_1 = h3.geo_to_h3(latitude, longitude, 1) #resolution 1
print(h3index_1)
h3index_2 = h3.geo_to_h3(latitude, longitude, 2) #resolution 2
print(h3index_2)

803ffffffffffff
813e7ffffffffff
823e67fffffffff


 - We can visualize this using **folium** package. From the map, we see that Suez Canal is contained within H3 '803ffffffffffff'

In [20]:
m = folium.Map(location=[latitude, longitude], zoom_start=4, tiles="cartodbpositron")
folium.Marker([latitude, longitude]).add_to(m)
folium.GeoJson(Polygon(h3.h3_to_geo_boundary(h3index_0, geo_json=True))).add_to(m)
folium.GeoJson(Polygon(h3.h3_to_geo_boundary(h3index_1, geo_json=True))).add_to(m)
folium.GeoJson(Polygon(h3.h3_to_geo_boundary(h3index_2, geo_json=True))).add_to(m)
m

# 4.2 filter data

Get the AIS records in side the H3 geo boundary

 - Resolution 0: 803ffffffffffff

In [21]:
#If we want to read all the records within this area. 
#We can use the spark.sql(), and cast the H3 index.
df1.createOrReplaceTempView("temp_df")
df_H3_0 = spark.sql("""
                select dt_pos_utc, mmsi, longitude, latitude, H3index_0, H3_int_index_0, H3_int_index_1, H3_int_index_2, H3_int_index_3
                from temp_df
                where H3index_0 = "803ffffffffffff"
                """)

In [22]:
df_H3_0.count()

np.int64(312825)

- Resolution 1: 813e7ffffffffff

In [23]:
h3index_1_int = int(h3index_1,16)
h3index_1_int

582063863558569983

In [24]:
df1.createOrReplaceTempView("temp_df")
df_H3_1 = spark.sql("""
                select dt_pos_utc, mmsi, longitude, latitude, H3index_0, H3_int_index_0, H3_int_index_1, H3_int_index_2, H3_int_index_3
                from temp_df
                where H3_int_index_1 = 582063863558569983
                """)

In [25]:
df_H3_1.count()

np.int64(36111)

# 5. Save and Download Data

## 5.1 Transform to DataFrame

In [26]:
dataframe_H3_1 = df_H3_1.toPandas()
dataframe_H3_1.head(10)

,dt_pos_utc,mmsi,longitude,latitude,H3index_0,H3_int_index_0,H3_int_index_1,H3_int_index_2,H3_int_index_3
0,2024-01-01 15:05:47,241166000,33.84,31.156667,803ffffffffffff,577586652210266111,582063863558569983,586272244313882623,590775775221776383
1,2024-01-01 15:14:49,241166000,33.84,31.156667,803ffffffffffff,577586652210266111,582063863558569983,586272244313882623,590775775221776383
2,2024-01-01 16:20:57,241166000,33.84,31.156667,803ffffffffffff,577586652210266111,582063863558569983,586272244313882623,590775775221776383
3,2024-01-01 16:08:47,241166000,33.84,31.156667,803ffffffffffff,577586652210266111,582063863558569983,586272244313882623,590775775221776383
4,2024-01-01 16:35:54,241166000,33.84,31.156667,803ffffffffffff,577586652210266111,582063863558569983,586272244313882623,590775775221776383
5,2024-01-01 14:05:46,241166000,33.84,31.156667,803ffffffffffff,577586652210266111,582063863558569983,586272244313882623,590775775221776383
6,2024-01-01 17:29:47,241166000,33.84,31.156667,803ffffffffffff,577586652210266111,582063863558569983,586272244313882623,590775775221776383
7,2024-01-01 16:44:50,241166000,33.84,31.156667,803ffffffffffff,577586652210266111,582063863558569983,586272244313882623,590775775221776383
8,2024-01-01 14:47:49,241166000,33.84,31.156667,803ffffffffffff,577586652210266111,582063863558569983,586272244313882623,590775775221776383
9,2024-01-01 16:11:55,241166000,33.84,31.156667,803ffffffffffff,577586652210266111,582063863558569983,586272244313882623,590775775221776383


## 5.2 Download Data

You can use create_download_link() function to download data. But please note that the load is limited. Do not try to download large dataset.

In [27]:
af.create_download_link(dataframe_H3_1.head(100), title = "Download CSV file", filename = "SampleResult.csv")

# 6.  Map for AIS data

In [28]:
import folium
from folium.plugins import HeatMap
from shapely.geometry import Polygon, MultiPoint, mapping
import h3

In [29]:
def create_geojson (h3_list):
    geojson = {"type": 'FeatureCollection',
               "features": [{ "type": "Feature",
                            "geometry": {"type":"Polygon",
                                         "coordinates": (h3.h3_to_geo_boundary(x,geo_json=True),)
                                        },
                            "properties": {"H3 index":x,"Resolution":h3.h3_get_resolution(x)}, 
                           } for x in h3_list]
               }
    return geojson

In [30]:
#In this example we choose the records near Suez Canal on Jan 1st 2024 as an example
#transform to Pandas format
df = df_H3_0.toPandas()
df

,dt_pos_utc,mmsi,longitude,latitude,H3index_0,H3_int_index_0,H3_int_index_1,H3_int_index_2,H3_int_index_3
0,2024-01-01 14:44:03,605076070,25.040000,39.040000,803ffffffffffff,577586652210266111,581509709698170879,586011110302285823,590514228893319167
1,2024-01-01 15:29:26,620604000,23.589033,38.995450,803ffffffffffff,577586652210266111,581509709698170879,586011110302285823,590514228893319167
2,2024-01-01 14:55:23,240184000,24.535000,38.841667,803ffffffffffff,577586652210266111,581509709698170879,586011110302285823,590514228893319167
3,2024-01-01 17:13:27,240184000,24.535000,38.841667,803ffffffffffff,577586652210266111,581509709698170879,586011110302285823,590514228893319167
4,2024-01-01 17:52:25,240184000,24.535000,38.841667,803ffffffffffff,577586652210266111,581509709698170879,586011110302285823,590514228893319167
...,...,...,...,...,...,...,...,...,...
312820,2024-01-01 22:03:28,240458000,23.640817,36.513667,803ffffffffffff,577586652210266111,582081455744614399,586584505616171007,591088036524064767
312821,2024-01-01 22:13:30,240458000,23.635217,36.489967,803ffffffffffff,577586652210266111,582081455744614399,586584505616171007,591088036524064767
312822,2024-01-01 23:00:27,368290790,16.956682,29.619002,803ffffffffffff,577586652210266111,582085853791125503,586588353906868223,591091678656331775
312823,2024-01-01 23:19:57,367351510,16.765073,29.787283,803ffffffffffff,577586652210266111,582085853791125503,586588353906868223,591091678656331775


# 6.1 Heat map

In [ ]:
m = folium.Map(location=[latitude, longitude], zoom_start=6, tiles="cartodbpositron")
a = HeatMap(df[['latitude','longitude']], radius=10, blur=10, name="AIS Data HeatMap").add_to(m)
m

# 6.2 Map data with H3 resolutions

In [ ]:
df['h3_1'] = df[['latitude','longitude']].apply(lambda x: h3.geo_to_h3(x[0],x[1],1), axis=1)
df['h3_2'] = df[['latitude','longitude']].apply(lambda x: h3.geo_to_h3(x[0],x[1],2), axis=1)
df['h3_3'] = df[['latitude','longitude']].apply(lambda x: h3.geo_to_h3(x[0],x[1],3), axis=1)

In [ ]:
a = folium.GeoJson(create_geojson(df["h3_1"].unique().tolist()),
                    tooltip = folium.GeoJsonTooltip(fields=['H3 index','Resolution']),
                    name='H3 Resolution 1', show=False).add_to(m)
a = folium.GeoJson(create_geojson(df["h3_2"].unique().tolist()),
                    tooltip = folium.GeoJsonTooltip(fields=['H3 index','Resolution']),
                    name='H3 Resolution 2', show=False).add_to(m)
a = folium.GeoJson(create_geojson(df["h3_3"].unique().tolist()),
                    tooltip = folium.GeoJsonTooltip(fields=['H3 index','Resolution']),
                    name='H3 Resolution 3', show=False).add_to(m)
folium.LayerControl().add_to(m)
m

# 5. Stop Spark Session

Always end the spark session to free any executors assigned to your tasks, and shut down the kernel.

In [ ]:
spark.stop()